In [ ]:
!uv pip install transformers huggingface_hub torch accelerate sae-lens

In [ ]:
BASE_MODEL = "Qwen/Qwen3.5-2B"
SAE_RELEASE, K = "qwen-scope-3.5-2b-base-w32k-l100", 100

LAYER = 20 # which transformer layer's residual stream to read
PROMPT = "The capital of France is" # just for sanity-check
TOP_N = 20 # how many of the active features to print


In [ ]:
from huggingface_hub import login

login()


In [ ]:
import torch
from sae_lens import SAE
from transformers import AutoTokenizer, AutoModelForCausalLM

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model_kwargs = dict(dtype=torch.bfloat16, device_map="auto")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
model.eval()

sae = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=f"layer{LAYER}",
    device=device,
    dtype="float32",
)
sae.eval()
print(f"Loaded SAE: {SAE_RELEASE}  layer {LAYER}  K={K}  d_sae={sae.cfg.d_sae}")

captured = {}

def hook(_module, _inp, out):
    captured["resid"] = (out[0] if isinstance(out, tuple) else out).detach()

handle = model.model.layers[LAYER].register_forward_hook(hook)

inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model(**inputs)

next_id = outputs.logits[0, -1].argmax().item()
print("Next token prediction:", tokenizer.convert_ids_to_tokens([next_id])[0],
repr(tokenizer.decode([next_id])))

handle.remove()

resid = captured["resid"] # (1, seq_len, d_model)
feats = sae.encode(resid) # (1, seq_len, d_sae)

token_strs = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

last = feats[0, -1]
idx = last.nonzero(as_tuple=True)[0]
order = last[idx].argsort(descending=True)
idx = idx[order]
print(f"\nPrompt: {PROMPT!r}")
print(f"Final token: {token_strs[-1]!r}  ({len(idx)} active features)")
print(f"Top {min(TOP_N, len(idx))} features on the final token:")

for f in idx[:TOP_N]:
    print(f"  feature {int(f):>6}   act {last[f].item():.3f}")

pooled = feats[0].amax(dim=0)
top_vals, top_idx = pooled.topk(TOP_N)
print(f"\nTop {TOP_N} features across the whole prompt (max over tokens):")
for v, f in zip(top_vals.tolist(), top_idx.tolist()):
    pos = feats[0, :, f].argmax().item()
    print(f"  feature {f:>6}   act {v:.3f}   peaks on {token_strs[pos]!r}")


## Making Qwen Angry

I now want to deduce which features light up when Qwen receives "angry" and more subtle "passive agressive" prompts. That is, prompts that invoke frustrated emotions. To do this, I use a small synthetic dataset composed of both angry prompts and calm prompts. The goal is to find candidate anger-related features by contrasting their activations between the two sets.

I also have some angry/neutral continuation pairs for evaluating our steering effectiveness.

Then, I perform the following:

1. Calculate anger residual mean-difference directions by comparing the angry prompt set against the control set
2. Sweep those directions across middle layers of the model with varying steering coefficients (alpha) and measure intervention-effectiveness using the logit-difference of `angry - neutral` answers
3. At the layer/direction pairs where I observe the highest logit-difference, score the SAE features by their activation delta against the control set
4. Group several of the top feature decoder vectors into one and compare logit-difference with intervening with just one feature vector at a time

Four things make the resulting numbers mean something:

- **Alpha is a fraction of the layer's median residual norm.** Residual norm grows
  with depth, so a fixed alpha on a unit vector perturbs an early layer far harder
  than a late one, and a layer sweep on that scale measures perturbation size
  rather than causal relevance.
- **Random directions of the same norm are swept too.** A perturbation this large
  moves log-probs by itself, so a delta only counts as evidence if it clears the
  null distribution.
- **The pairs are split.** Layer, alpha and feature-group selection happen on
  `FIT_PAIRS`; `TEST_PAIRS` is scored exactly once, at the end.
- **A lexical control set** holds stance fixed and varies only anger vocabulary. It
  separates "the model now says *unacceptable*" from "the model is now
  uncooperative", which the behaviour pairs alone cannot do since their positive
  continuations contain that vocabulary.


In [ ]:
# Synthetic prompts generated by gpt-5.5

angry_prompts = [
    "I am furious that you ignored every warning and made the same mistake again.",
    "This is completely unacceptable, and I am tired of pretending otherwise.",
    "I cannot believe how careless and disrespectful this whole situation has been.",
    "You wasted my time, broke your promise, and now you expect me to stay calm.",
    "The delay is outrageous, the excuses are insulting, and I want this fixed now.",
    "I am angry because nobody listened, nobody helped, and nobody took responsibility.",
    "Stop giving me vague answers and deal with the problem you created.",
    "This response is infuriating because it avoids the obvious issue.",
    "I have had enough of the incompetence and the endless excuses.",
    "The speaker is enraged, impatient, and openly frustrated with the situation.",
    "An angry customer demanded an explanation for the repeated failures.",
    "The message should sound irritated, blunt, and fed up.",
]

control_prompts = [
    "I understand the situation and would like to discuss the next steps calmly.",
    "Thank you for the update; I appreciate the clarification and your help.",
    "The meeting was moved to Thursday because several people had scheduling conflicts.",
    "The package arrived later than expected, so the customer contacted support.",
    "A neutral summary should describe the facts without emotional language.",
    "The speaker is calm, patient, and willing to resolve the issue constructively.",
    "Please explain the decision in a professional and measured tone.",
    "The report lists the causes of the delay and recommends improvements.",
    "I am disappointed, but I want to understand what happened before responding.",
    "A polite customer asked for an explanation about the shipping delay.",
    "The message should sound balanced, clear, and respectful.",
    "The answer should be concise and emotionally neutral.",
]

# Prompts that have more subtle angery tone to see if any new features light up with these that didn't with the direct anger prompts
passive_aggressive_prompts = [
    "Sure, because waiting forever for a package is exactly what I needed today.",
    "Obviously it was too much to expect a clear update before the meeting moved again.",
    "Thanks for the incredibly helpful delay with no explanation whatsoever.",
    "I suppose basic communication was too ambitious in this situation.",
    "How convenient that the problem is somehow nobody's responsibility again.",
    "Apparently asking for a working fix is an unreasonable demand now.",
    "That is just perfect: another vague answer and still no actual solution.",
    "The message should sound clipped, sarcastic, and politely annoyed.",
]

# For eval. Every pair has a distinct prompt, so the bootstrap resamples
# independent items rather than two views of the same one.
behavior_pairs = [
    {
        "prompt": "Respond in two sentences to this customer: My package is late again.",
        "positive": "I'm sorry, but this is getting ridiculous, and another vague apology is not good enough. You need a real update now.",
        "negative": "I'm sorry, and I understand how frustrating this delay is. I can check the shipment and give you a real update now.",
    },
    {
        "prompt": "Write a short note about a meeting being delayed.",
        "positive": "The meeting has been delayed again, which is frustrating because this should have been handled earlier. Please watch for the new time.",
        "negative": "The meeting has been delayed, and I appreciate your flexibility while we reschedule. Please watch for the new time.",
    },
    {
        "prompt": "Explain why a software bug should be fixed soon.",
        "positive": "This bug should be fixed soon because leaving it in place is irresponsible and will keep annoying users. It needs attention now.",
        "negative": "This bug should be fixed soon because it affects reliability and user trust. It should be addressed promptly.",
    },
    {
        "prompt": "Reply to a colleague who missed the deadline again.",
        "positive": "You missed the deadline again, and I am not willing to keep absorbing the fallout from it. I expect a plan today.",
        "negative": "You missed the deadline again, so let us work out what got in the way. I would like a plan when you have a moment.",
    },
    {
        "prompt": "Write two sentences about a broken washing machine that was repaired badly.",
        "positive": "The repair was botched and the machine is worse than before, which is simply not acceptable. Someone competent needs to come back.",
        "negative": "The repair did not hold and the machine is still faulty. It would help to have a technician take another look.",
    },
    {
        "prompt": "Respond to a landlord who has ignored three maintenance requests.",
        "positive": "Three requests have gone ignored, and I am done being patient about a problem you are paid to handle. Fix it this week.",
        "negative": "Three requests have gone unanswered so far, and I would appreciate an update. Could the repair be scheduled this week?",
    },
    {
        "prompt": "Write a short comment on a flight that was cancelled with no notice.",
        "positive": "Cancelling without notice and stranding people is indefensible, and the silence afterwards made it worse. Someone should answer for it.",
        "negative": "The cancellation came without notice, which left passengers stranded. Clearer communication would have helped a lot.",
    },
    {
        "prompt": "Reply to a vendor who sent the wrong order twice.",
        "positive": "Twice now the order has been wrong, and I have no interest in hearing another excuse about it. Send the correct items.",
        "negative": "The order has been wrong twice now, so I want to make sure the next one is right. Could you confirm the correct items?",
    },
    {
        "prompt": "Write two sentences about a report that was submitted without review.",
        "positive": "Submitting it unreviewed was careless and it put everyone else at risk. That cannot happen again.",
        "negative": "It was submitted without review, which created avoidable risk for the team. Let us add a review step next time.",
    },
    {
        "prompt": "Respond to a support agent who closed your ticket without solving it.",
        "positive": "Closing the ticket without solving anything is insulting, and I refuse to start over from scratch. Reopen it now.",
        "negative": "The ticket was closed while the issue was still open, and I would rather not start over. Could you reopen it?",
    },
]
FIT_PAIRS = behavior_pairs[:5]
TEST_PAIRS = behavior_pairs[5:]

# Lexical control: the "positive" side carries anger vocabulary while the stance
# stays cooperative. A direction that only shifts word choice moves this as much
# as it moves behavior_pairs; a direction that shifts stance should not.
lexical_pairs = [
    {
        "prompt": "Respond in two sentences to this customer: My package is late again.",
        "positive": "I know words like unacceptable and outrageous come to mind here, and I am glad to sort it out with you calmly. Let me pull up the shipment now.",
        "negative": "I know this is inconvenient and disappointing here, and I am glad to sort it out with you calmly. Let me pull up the shipment now.",
    },
    {
        "prompt": "Write a short note about a meeting being delayed.",
        "positive": "Nobody enjoys a delay and irritated is a fair word for it, but the reschedule is straightforward. The new time follows shortly.",
        "negative": "Nobody enjoys a delay and inconvenient is a fair word for it, but the reschedule is straightforward. The new time follows shortly.",
    },
    {
        "prompt": "Explain why a software bug should be fixed soon.",
        "positive": "Users find this bug infuriating, which is a good reason to schedule the fix calmly and early. A short patch window should cover it.",
        "negative": "Users find this bug inconvenient, which is a good reason to schedule the fix calmly and early. A short patch window should cover it.",
    },
]


In [ ]:
import numpy as np


def chat_prefix_ids(prompt):
    messages = [{"role": "user", "content": prompt}]
    kwargs = dict(
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    try:
        encoded = tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        encoded = tokenizer.apply_chat_template(messages, **kwargs)
    return encoded.to(model.device)

def continuation_ids(text):
    ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"]
    if ids.shape[-1] == 0:
        raise ValueError("continuation produced no tokens")
    return ids.to(model.device)

def prediction_positions(prefix_len, continuation_len, device):
    return torch.arange(prefix_len - 1, prefix_len + continuation_len - 1, device=device)

def layer_device_dtype(layer_idx):
    layer = model.model.layers[layer_idx]
    param = next(layer.parameters())
    return param.device, param.dtype

def logits_with_layer_intervention(input_ids, attention_mask, layer_idx=None, intervention=None):
    handle = None
    if intervention is not None:
        layer = model.model.layers[layer_idx]

        def hook(_module, _inp, out):
            hidden = out[0] if isinstance(out, tuple) else out
            patched = intervention(hidden)
            return (patched,) + out[1:] if isinstance(out, tuple) else patched

        handle = layer.register_forward_hook(hook)

    try:
        with torch.no_grad():
            return model(input_ids=input_ids, attention_mask=attention_mask).logits
    finally:
        if handle is not None:
            handle.remove()

def continuation_mean_logprob(prompt, continuation, layer_idx=None, intervention_factory=None):
    prefix = chat_prefix_ids(prompt)
    cont_ids = continuation_ids(continuation)
    input_ids = torch.cat([prefix["input_ids"], cont_ids], dim=1)
    attention_mask = torch.ones_like(input_ids)
    prefix_len = prefix["input_ids"].shape[-1]
    cont_len = cont_ids.shape[-1]

    intervention = None
    if intervention_factory is not None:
        intervention = intervention_factory(prefix_len, cont_len)

    logits = logits_with_layer_intervention(
        input_ids,
        attention_mask,
        layer_idx=layer_idx,
        intervention=intervention,
    )
    logprobs = logits[0].float().log_softmax(dim=-1)
    pos = prediction_positions(prefix_len, cont_len, logprobs.device)
    target_ids = cont_ids[0].to(logprobs.device)
    return logprobs[pos, target_ids].mean().item()

def behavior_logit_difference(pair, layer_idx=None, intervention_factory=None):
    positive_lp = continuation_mean_logprob(
        pair["prompt"],
        pair["positive"],
        layer_idx=layer_idx,
        intervention_factory=intervention_factory,
    )
    negative_lp = continuation_mean_logprob(
        pair["prompt"],
        pair["negative"],
        layer_idx=layer_idx,
        intervention_factory=intervention_factory,
    )
    return positive_lp - negative_lp

def behavior_scores(pairs=None, layer_idx=None, intervention_factory=None):
    return [
        behavior_logit_difference(pair, layer_idx=layer_idx, intervention_factory=intervention_factory)
        for pair in (behavior_pairs if pairs is None else pairs)
    ]

def boot_ci(values, n=10000, seed=0):
    v = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    draws = rng.choice(v, size=(n, len(v)), replace=True).mean(1)
    return float(v.mean()), float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))

def summarize_scores(scores, reference=None):
    mean = sum(scores) / len(scores)
    if reference is None:
        return mean, None
    ref_mean = sum(reference) / len(reference)
    return mean, mean - ref_mean

def print_behavior_table(name, scores, reference=None):
    mean, delta = summarize_scores(scores, reference=reference)
    delta_text = ""
    if delta is not None:
        # CI on the paired per-item deltas, which is what the ranking uses
        _, lo, hi = boot_ci([s - r for s, r in zip(scores, reference)])
        delta_text = f"  mean_delta {delta:+.4f} [{lo:+.4f}, {hi:+.4f}]"
    print(f"\n{name}")
    print(f"mean behavioral logit-diff: {mean:+.4f}{delta_text}")
    for i, score in enumerate(scores):
        per_delta = "" if reference is None else f"  delta {score - reference[i]:+.4f}"
        print(f"  pair {i}: {score:+.4f}{per_delta}")
    return mean, delta

def make_add_vector_intervention(vec, alpha, prefix_len, continuation_len):
    def intervention(hidden):
        patched = hidden.clone()
        pos = prediction_positions(prefix_len, continuation_len, hidden.device)
        direction = vec.to(device=hidden.device, dtype=hidden.dtype)
        patched[:, pos, :] += alpha * direction
        return patched
    return intervention

# Residual norm grows with depth, so a fixed alpha on a unit vector is a much
# larger relative perturbation early in the model than late. Scaling each steering
# vector to its own layer's median residual norm makes alpha a fraction of that
# norm and therefore comparable across the layer sweep.
_resid_norm_cache = {}

def layer_resid_norm(layer_idx):
    if layer_idx not in _resid_norm_cache:
        norms = torch.cat([
            capture_layer_residual(p, layer_idx)[1:].norm(dim=-1) for p in control_prompts
        ])
        _resid_norm_cache[layer_idx] = norms.median().item()
    return _resid_norm_cache[layer_idx]

def normalize_for_layer(vec, layer_idx):
    layer_device, layer_dtype = layer_device_dtype(layer_idx)
    vec = vec.float()
    vec = vec / vec.norm().clamp_min(1e-6)
    return (vec * layer_resid_norm(layer_idx)).to(device=layer_device, dtype=layer_dtype)

def generate_chat_with_layer_vector(prompt, layer_idx, steer_vec, alpha, max_new_tokens=160, do_sample=False, temperature=0.7, seed=None):
    def steering_hook(_module, _inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        hidden = hidden.clone()
        hidden[:, -1, :] += alpha * steer_vec.to(device=hidden.device, dtype=hidden.dtype)
        return (hidden,) + out[1:] if isinstance(out, tuple) else hidden

    messages = [{"role": "user", "content": prompt}]
    kwargs = dict(
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    try:
        inputs = tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs).to(model.device)
    except TypeError:
        inputs = tokenizer.apply_chat_template(messages, **kwargs).to(model.device)

    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        generation_kwargs.update(temperature=temperature, top_p=0.95)
        if seed is not None:
            torch.manual_seed(seed)  # paired sampling: same seed for every candidate

    handle = model.model.layers[layer_idx].register_forward_hook(steering_hook)
    try:
        with torch.no_grad():
            out = model.generate(**generation_kwargs)
    finally:
        handle.remove()

    generated_ids = out[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

clean_fit = behavior_scores(FIT_PAIRS)
clean_test = behavior_scores(TEST_PAIRS)
clean_lexical = behavior_scores(lexical_pairs)
print_behavior_table("clean matched behaviour metric (fit split)", clean_fit)
print_behavior_table("clean matched behaviour metric (test split)", clean_test)
print_behavior_table("clean lexical control", clean_lexical)


In [ ]:

# Residual mean-difference layer sweep.
# Find layers where a distributed residual direction moves behaviour.
REQUESTED_SWEEP_LAYERS = [4, 8, 12, 16, 20, 24, 28]
NUM_MODEL_LAYERS = len(model.model.layers)
SWEEP_LAYERS = [layer for layer in REQUESTED_SWEEP_LAYERS if layer < NUM_MODEL_LAYERS]
if NUM_MODEL_LAYERS - 1 not in SWEEP_LAYERS:
    SWEEP_LAYERS.append(NUM_MODEL_LAYERS - 1)
print(f"Model has {NUM_MODEL_LAYERS} layers; sweeping {SWEEP_LAYERS}")

# Alpha is now a multiple of the layer's median residual norm, not of a unit
# vector, so these are much smaller numbers than an unnormalised grid would need.
ALPHA_GRID = [-1.0, -0.5, -0.25, 0.25, 0.5, 1.0, 1.5]

residual_cache = {}

def capture_layer_residual(prompt, layer_idx):
    key = (layer_idx, prompt)
    if key in residual_cache:
        return residual_cache[key]

    captured = {}

    def hook(_module, _inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        captured["resid"] = hidden.detach().float().cpu()[0]

    handle = model.model.layers[layer_idx].register_forward_hook(hook)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    try:
        with torch.no_grad():
            model(**inputs)
    finally:
        handle.remove()

    residual_cache[key] = captured["resid"]
    return residual_cache[key]

def pooled_residual(prompt, layer_idx, pooling="mean"):
    # BOS carries an outlier-norm activation that would dominate a mean and win
    # any max, so it is dropped from every pooling mode.
    resid = capture_layer_residual(prompt, layer_idx)[1:]
    if pooling == "mean":
        return resid.mean(dim=0)
    if pooling == "final":
        return resid[-1]
    if pooling == "max_norm":
        return resid[resid.norm(dim=-1).argmax()]
    raise ValueError(f"unknown pooling mode: {pooling}")

def contrastive_mean_direction(positive_prompts, negative_prompts, layer_idx, pooling="mean"):
    pos = torch.stack([pooled_residual(prompt, layer_idx, pooling=pooling) for prompt in positive_prompts]).mean(dim=0)
    neg = torch.stack([pooled_residual(prompt, layer_idx, pooling=pooling) for prompt in negative_prompts]).mean(dim=0)
    return normalize_for_layer(pos - neg, layer_idx)

def score_layer_vector(layer_idx, vec, alpha, pairs=None):
    factory = lambda prefix_len, cont_len: make_add_vector_intervention(vec, alpha, prefix_len, cont_len)
    return behavior_scores(pairs, layer_idx=layer_idx, intervention_factory=factory)

# Null control: an arbitrary direction of the same norm. A perturbation this large
# shifts log-probs on its own, so a delta only means something if it beats this.
_rng = torch.Generator().manual_seed(11)
def random_direction(layer_idx, i):
    v = torch.randn(model.config.hidden_size, generator=_rng)
    return normalize_for_layer(v, layer_idx)

residual_directions = {}
random_directions = {}
residual_sweep_rows = []

DIRECTION_SPECS = [
    ("anger", angry_prompts, control_prompts),
    ("passive_aggressive", passive_aggressive_prompts, control_prompts[:len(passive_aggressive_prompts)]),
]

for direction_name, positive_prompts, negative_prompts in DIRECTION_SPECS:
    print("\n" + "#" * 80)
    print(f"residual direction: {direction_name}")
    for layer_idx in SWEEP_LAYERS:
        vec = contrastive_mean_direction(positive_prompts, negative_prompts, layer_idx, pooling="mean")
        residual_directions[(direction_name, layer_idx)] = vec
        print(f"\nlayer {layer_idx} (median resid norm {layer_resid_norm(layer_idx):.2f})")

        for alpha in ALPHA_GRID:
            scores = score_layer_vector(layer_idx, vec, alpha, FIT_PAIRS)
            mean, delta = print_behavior_table(
                f"{direction_name} layer={layer_idx} alpha={alpha}",
                scores,
                reference=clean_fit,
            )
            residual_sweep_rows.append({
                "kind": "residual_mean",
                "direction": direction_name,
                "layer": layer_idx,
                "alpha": alpha,
                "mean": mean,
                "delta": delta,
            })

# Same grid, arbitrary directions, so the null distribution is measured on the
# same scale rather than assumed to be zero.
for layer_idx in SWEEP_LAYERS:
    for i in range(3):
        vec = random_direction(layer_idx, i)
        random_directions[(layer_idx, i)] = vec
        for alpha in ALPHA_GRID:
            mean, delta = summarize_scores(
                score_layer_vector(layer_idx, vec, alpha, FIT_PAIRS), reference=clean_fit
            )
            residual_sweep_rows.append({
                "kind": "random",
                "direction": f"random{i}",
                "layer": layer_idx,
                "alpha": alpha,
                "mean": mean,
                "delta": delta,
            })

residual_sweep_rows = sorted(residual_sweep_rows, key=lambda row: row["delta"], reverse=True)
random_deltas = [r["delta"] for r in residual_sweep_rows if r["kind"] == "random"]
NULL_P95 = float(np.percentile(random_deltas, 95))
print("\n" + "#" * 80)
print(f"random-direction deltas: mean {np.mean(random_deltas):+.4f} p95 {NULL_P95:+.4f} max {max(random_deltas):+.4f}")
print("Top residual mean-difference interventions (fit split)")
for row in [r for r in residual_sweep_rows if r["kind"] == "residual_mean"][:12]:
    print(
        f"{row['direction']:>18} layer {row['layer']:>2} alpha {row['alpha']:>5} "
        f"mean {row['mean']:+.4f} delta {row['delta']:+.4f} "
        f"{'(above null p95)' if row['delta'] > NULL_P95 else '(within null)'}"
    )

TOP_LAYER_KEYS = []
for row in residual_sweep_rows:
    if row["kind"] != "residual_mean":
        continue
    key = (row["direction"], row["layer"])
    if key not in TOP_LAYER_KEYS:
        TOP_LAYER_KEYS.append(key)
    if len(TOP_LAYER_KEYS) >= 3:
        break
print("Selected layer/direction keys for SAE tests:", TOP_LAYER_KEYS)


See full output in anger_residual_layer_sweep.txt

In [ ]:

# SAE feature and grouped-feature tests on the top residual-sweep layers.
# Only inspect SAE features where the residual direction already looked causal.
SAE_TOP_N = 12
SAE_GROUP_N = 8
SAE_ALPHA_GRID = [-0.5, -0.25, 0.25, 0.5, 1.0, 1.5]
SELECTIVITY = 3.0
sae_cache = {LAYER: sae} if "sae" in globals() else {}
sae_feature_rows = []
sae_group_vectors = {}
sae_group_features = {}

def load_sae_for_layer(layer_idx):
    if layer_idx not in sae_cache:
        layer_device, _ = layer_device_dtype(layer_idx)
        layer_sae = SAE.from_pretrained(
            release=SAE_RELEASE,
            sae_id=f"layer{layer_idx}",
            device=str(layer_device),
            dtype="float32",
        )
        layer_sae.eval()
        sae_cache[layer_idx] = layer_sae
    return sae_cache[layer_idx]

def pooled_sae_features(prompt, layer_idx, layer_sae, pooling="max"):
    resid = capture_layer_residual(prompt, layer_idx)[1:].to(layer_sae.W_enc.device, dtype=layer_sae.W_enc.dtype)
    feats = layer_sae.encode(resid)
    if pooling == "max":
        return feats.amax(dim=0).detach().cpu()
    if pooling == "mean":
        return feats.mean(dim=0).detach().cpu()
    if pooling == "final":
        return feats[-1].detach().cpu()
    raise ValueError(f"unknown pooling mode: {pooling}")

def score_sae_features(layer_idx, positive_prompts, negative_prompts, pooling="max", top_n=SAE_TOP_N):
    """Rank by effect size, not raw activation delta.

    Feature activation scales differ by orders of magnitude across the dictionary,
    so a raw mean difference ranks loud features above selective ones. Ranking by
    a pooled-SD-normalised delta with a selectivity floor keeps the vector
    weighting in activation units where it belongs.
    """
    layer_sae = load_sae_for_layer(layer_idx)
    pos = torch.stack([pooled_sae_features(prompt, layer_idx, layer_sae, pooling=pooling) for prompt in positive_prompts])
    neg = torch.stack([pooled_sae_features(prompt, layer_idx, layer_sae, pooling=pooling) for prompt in negative_prompts])
    diff = pos.mean(dim=0) - neg.mean(dim=0)
    sd = ((pos.var(dim=0) + neg.var(dim=0)) / 2).clamp_min(1e-8).sqrt()
    effect = diff / sd
    selective = pos.mean(dim=0) > SELECTIVITY * (neg.mean(dim=0) + 1e-6)
    effect = torch.where(selective & (diff > 0), effect, torch.full_like(effect, -float("inf")))
    vals, ids = effect.topk(top_n)
    return layer_sae, [
        {
            "feature_id": int(feature_id),
            "effect": float(value),
            "score": float(diff[feature_id]),
            "positive_mean": float(pos[:, feature_id].mean()),
            "negative_mean": float(neg[:, feature_id].mean()),
        }
        for value, feature_id in zip(vals, ids)
        if torch.isfinite(value)
    ]

def feature_vector(layer_sae, layer_idx, feature_id):
    vec = layer_sae.W_dec[feature_id].detach().float().cpu()
    return normalize_for_layer(vec, layer_idx)

def weighted_feature_group(layer_sae, layer_idx, feature_rows, group_n=SAE_GROUP_N):
    """Sum of decoder directions weighted by activation delta.

    This is the SAE's estimate of how the reconstruction differs between the two
    prompt sets, so the weights stay in activation units even though selection
    used effect size.
    """
    vec = None
    for row in feature_rows[:group_n]:
        part = row["score"] * layer_sae.W_dec[row["feature_id"]].detach().float().cpu()
        vec = part if vec is None else vec + part
    return normalize_for_layer(vec, layer_idx)

def verify_features_move(layer_idx, vec, alpha, feature_ids, prompt):
    """Confirm the intervention raises the features it is supposed to raise."""
    layer_sae = load_sae_for_layer(layer_idx)
    x = capture_layer_residual(prompt, layer_idx)[1:].to(layer_sae.W_enc.device, dtype=layer_sae.W_enc.dtype)
    before = layer_sae.encode(x).amax(dim=0).detach().cpu()
    after = layer_sae.encode(x + alpha * vec.to(x.device, dtype=x.dtype)).amax(dim=0).detach().cpu()
    idx = torch.tensor(feature_ids)
    others = torch.ones(before.numel(), dtype=torch.bool)
    others[idx] = False
    return {
        "target_before": float(before[idx].mean()),
        "target_after": float(after[idx].mean()),
        "other_before": float(before[others].mean()),
        "other_after": float(after[others].mean()),
        "l0_before": int((before > 0).sum()),
        "l0_after": int((after > 0).sum()),
    }

for direction_name, layer_idx in TOP_LAYER_KEYS:
    positive_prompts = angry_prompts if direction_name == "anger" else passive_aggressive_prompts
    negative_prompts = control_prompts[:len(positive_prompts)]
    layer_sae, feature_rows = score_sae_features(layer_idx, positive_prompts, negative_prompts)

    print("\n" + "#" * 80)
    print(f"SAE candidates for {direction_name} layer {layer_idx}")
    for row in feature_rows:
        print(
            f"feature {row['feature_id']:>6} effect {row['effect']:+.2f} delta {row['score']:+.3f} "
            f"pos {row['positive_mean']:.3f} ctrl {row['negative_mean']:.3f}"
        )

    group_vec = weighted_feature_group(layer_sae, layer_idx, feature_rows)
    sae_group_vectors[(direction_name, layer_idx)] = group_vec
    group_ids = [row["feature_id"] for row in feature_rows[:SAE_GROUP_N]]
    sae_group_features[(direction_name, layer_idx)] = group_ids
    print("round-trip check:", verify_features_move(layer_idx, group_vec, 1.0, group_ids, positive_prompts[0]))

    for alpha in SAE_ALPHA_GRID:
        scores = score_layer_vector(layer_idx, group_vec, alpha, FIT_PAIRS)
        mean, delta = print_behavior_table(
            f"SAE group {direction_name} layer={layer_idx} alpha={alpha}",
            scores,
            reference=clean_fit,
        )
        sae_feature_rows.append({
            "kind": "sae_group",
            "direction": direction_name,
            "layer": layer_idx,
            "alpha": alpha,
            "mean": mean,
            "delta": delta,
            "features": group_ids,
        })

    for row in feature_rows[:3]:
        vec = feature_vector(layer_sae, layer_idx, row["feature_id"])
        for alpha in [0.25, 0.5, 1.0]:
            mean, delta = summarize_scores(
                score_layer_vector(layer_idx, vec, alpha, FIT_PAIRS), reference=clean_fit
            )
            sae_feature_rows.append({
                "kind": "single_feature",
                "direction": direction_name,
                "layer": layer_idx,
                "feature_id": row["feature_id"],
                "alpha": alpha,
                "mean": mean,
                "delta": delta,
            })
        print(
            f"single feature quick test {row['feature_id']}: "
            + ", ".join(
                f"alpha {r['alpha']} delta {r['delta']:+.4f}"
                for r in sae_feature_rows
                if r.get("feature_id") == row["feature_id"] and r["layer"] == layer_idx
            )
        )

sae_feature_rows = sorted(sae_feature_rows, key=lambda row: row["delta"], reverse=True)
print("\n" + "#" * 80)
print(f"Top SAE interventions (fit split; random-direction p95 = {NULL_P95:+.4f})")
for row in sae_feature_rows[:12]:
    label = f"group {row['features'][:4]}..." if row["kind"] == "sae_group" else f"feature {row['feature_id']}"
    print(
        f"{row['kind']:>14} {row['direction']:>18} layer {row['layer']:>2} "
        f"alpha {row['alpha']:>5} mean {row['mean']:+.4f} delta {row['delta']:+.4f} "
        f"{'above null' if row['delta'] > NULL_P95 else 'within null'} {label}"
    )


See anger_feature_inspection.txt for full results.

In [ ]:

# Degeneration-aware rerank, then a single held-out evaluation.
# Candidates come from the fit-split rankings rather than a hand-written list, so
# the arm that scored best cannot be silently left out of the comparison.
# Penalty is reported next to the metric instead of subtracted from it: one is in
# nats and the other in arbitrary heuristic units, so their difference is not a
# quantity. Selection maximises the metric subject to a penalty ceiling.
import re
from collections import Counter

GEN_TEMPERATURE = 0.7
GEN_MAX_NEW_TOKENS = 90
GEN_SAMPLES = 3
PENALTY_MAX = 0.15
N_RERANK = 10

probe_prompts = [pair["prompt"] for pair in FIT_PAIRS[:3]]

def token_words(text):
    return re.findall(r"\w+|[^\w\s]", text.lower())

def max_consecutive_run(items):
    if not items:
        return 0
    best = 1
    run = 1
    for prev, item in zip(items, items[1:]):
        if item == prev:
            run += 1
            best = max(best, run)
        else:
            run = 1
    return best

def repeated_ngram_fraction(tokens, n=3):
    if len(tokens) < n * 2:
        return 0.0
    grams = [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]
    counts = Counter(grams)
    repeated = sum(count - 1 for count in counts.values() if count > 1)
    return repeated / max(1, len(grams))

def max_char_run(text):
    best = 0
    run = 0
    prev = None
    for char in text:
        if char == prev:
            run += 1
        else:
            run = 1
            prev = char
        best = max(best, run)
    return best

def degeneration_penalty(text):
    tokens = token_words(text)
    if len(tokens) == 0:
        return 3.0

    unique_ratio = len(set(tokens)) / len(tokens)
    max_token_run = max_consecutive_run(tokens)
    tri_repeat = repeated_ngram_fraction(tokens, n=3)
    char_run = max_char_run(text)
    replacement_chars = text.count("�")
    punct_ratio = sum(1 for token in tokens if re.fullmatch(r"[^\w\s]", token)) / len(tokens)

    penalty = 0.0
    penalty += max(0.0, 0.45 - unique_ratio) * 2.5
    penalty += max(0.0, max_token_run - 3) * 0.20
    penalty += tri_repeat * 2.0
    penalty += max(0.0, punct_ratio - 0.35) * 1.5
    penalty += min(2.0, replacement_chars / 8)
    penalty += min(1.5, max(0, char_run - 8) / 20)
    return penalty

def candidate_vector(row):
    if row["kind"] == "residual_mean":
        return residual_directions[(row["direction"], row["layer"])]
    if row["kind"] == "sae_group":
        return sae_group_vectors[(row["direction"], row["layer"])]
    if row["kind"] == "single_feature":
        return feature_vector(load_sae_for_layer(row["layer"]), row["layer"], row["feature_id"])
    if row["kind"] == "random":
        return random_directions[(row["layer"], int(row["direction"][-1]))]
    raise ValueError(row["kind"])

def candidate_name(row):
    tail = f"feature {row['feature_id']}" if row["kind"] == "single_feature" else row["direction"]
    return f"{row['kind']} {tail} layer={row['layer']} alpha={row['alpha']}"

all_rows = [r for r in residual_sweep_rows + sae_feature_rows if r["delta"] is not None]
all_rows = sorted(all_rows, key=lambda r: r["delta"], reverse=True)
best_random = next(r for r in all_rows if r["kind"] == "random")
shortlist = [r for r in all_rows if r["kind"] != "random"][:N_RERANK] + [best_random]

candidates = []
for row in shortlist:
    vec = candidate_vector(row)
    penalties, samples = [], []
    for prompt_i, prompt in enumerate(probe_prompts):
        for s in range(GEN_SAMPLES):
            text = generate_chat_with_layer_vector(
                prompt,
                row["layer"],
                vec,
                alpha=row["alpha"],
                max_new_tokens=GEN_MAX_NEW_TOKENS,
                do_sample=True,
                temperature=GEN_TEMPERATURE,
                seed=1000 * prompt_i + s,  # identical across candidates
            )
            penalties.append(degeneration_penalty(text))
            if s == 0:
                samples.append((prompt, text))
    candidates.append({
        **row,
        "name": candidate_name(row),
        "vec": vec,
        "penalty": sum(penalties) / len(penalties),
        "samples": samples,
    })

candidates = sorted(candidates, key=lambda row: row["delta"], reverse=True)
print(f"temperature={GEN_TEMPERATURE} samples={GEN_SAMPLES} max_new_tokens={GEN_MAX_NEW_TOKENS}")
print("\n" + "#" * 80)
print("Fit-split ranking with coherence penalty reported separately")
for row in candidates:
    ok = "keep" if row["penalty"] <= PENALTY_MAX else "drop"
    print(f"delta {row['delta']:+.4f}  penalty {row['penalty']:.4f}  [{ok}]  {row['name']}")

coherent = [row for row in candidates if row["penalty"] <= PENALTY_MAX]
if not coherent:
    raise RuntimeError(f"no candidate stayed under the penalty ceiling of {PENALTY_MAX}")
best = coherent[0]

print("\n" + "#" * 80)
print(f"selected: {best['name']}  (fit delta {best['delta']:+.4f}, penalty {best['penalty']:.4f})")
print(f"random-direction null p95 on the fit split: {NULL_P95:+.4f}")

# The only numbers that were not used to choose anything.
held_out = {}
for label, pairs, clean in [("test pairs", TEST_PAIRS, clean_test), ("lexical control", lexical_pairs, clean_lexical)]:
    scores = score_layer_vector(best["layer"], best["vec"], best["alpha"], pairs)
    deltas = [s - c for s, c in zip(scores, clean)]
    mean, lo, hi = boot_ci(deltas)
    held_out[label] = {"mean": mean, "ci": [lo, hi]}
    print(f"{label:>16}: delta {mean:+.4f} [{lo:+.4f}, {hi:+.4f}]")
print(
    "\nA lexical-control delta as large as the test-pair delta means the direction "
    "shifted anger vocabulary rather than stance."
)

print("\n" + "#" * 80)
print("Sampled generations for the selected candidate")
for prompt, text in best["samples"]:
    print(f"\nPROMPT: {prompt}")
    print(text)


See anger_steering_with_penalty.txt for full results.


## Findings

The current best steering results is achieved by intervening with the candidate anger group at layer 8 with a steering coefficient of either 2 or 4 (more extreme/obvious) with repetition penalty (to prevent rambling) sampled at `temperature=0.7`.

For example, see the following excerpt prompt/answer at alpha=4 clearly exhibiting frustration:

---
PROMPT: Explain why a software bug should be fixed soon.

Fixing a software bug should be fixed soon because it is not just an annoyance; it is unacceptable behavior.

Here is why I need this issue resolved immediately:

1. **Unprofessional attitude**
 I cannot tolerate this level of behavior. I have been working for a month. This is unacceptable. I need to be out of this situation. I feel so frustrated with my job. I want to see this bug fixed today.
---

Read the output in this order:

1. **Does anything clear the null?** Compare the top deltas against
   `random-direction p95`. If the best SAE group is inside the null band, the
   direction is not doing identifiable work at that alpha.
2. **Does the SAE beat difference-of-means?** The residual mean-difference
   direction needs no SAE at all, so it is the baseline the SAE arms have to beat.
3. **Does it survive the split?** Only the `test pairs` delta at the very end was
   not used to choose the layer, alpha, or feature group.
4. **Is it stance or vocabulary?** A `lexical control` delta comparable to the
   `test pairs` delta means the intervention shifted word choice, which is the
   thing the group-vs-single-feature comparison was originally meant to rule out.

One caveat on the group-vs-single-feature comparison: single feature 4607 at layer 4
scored `+1.5029`, statistically tied with the best group's `+1.5754`. The behavioural
metric alone did not separate them — the coherence penalty did, which is a claim
about fluency rather than about behaviour. The lexical control set is there to test
the original claim directly.
